# AI-Powered Network Intrusion Detection System

This notebook will be focused into training the model so we can implement it in the actual program.

In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical

In [34]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'network-intrusion-dataset' dataset.
Path to dataset files: /kaggle/input/network-intrusion-dataset


## Dataset Preparation (TODO)

Prepare the "kagglehub.dataset_download("chethuhn/network-intrusion-dataset")" dataset for it to be used at training a keras model.

In [35]:
# Load CSV
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
print("CSV files:", csv_files)

df = pd.read_csv(os.path.join(path, csv_files[0]))

print("Dataset shape:", df.shape)
print(df.head())

CSV files: ['Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'Monday-WorkingHours.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv']
Dataset shape: (286467, 79)
    Destination Port   Flow Duration   Total Fwd Packets  \
0                 22         1266342                  41   
1                 22         1319353                  41   
2                 22             160                   1   
3                 22         1303488                  41   
4              35396              77                   1   

    Total Backward Packets  Total Length of Fwd Packets  \
0                       44                         2664   
1                       44                         2664   
2                        1       

In [36]:
# remove whitespaces from column names
df.columns = df.columns.str.strip()

In [37]:
# Inspect Columns
print("Columns:")
print(df.columns.tolist())

Columns:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE F

In [38]:
# Drop duplicates and missing
df = df.drop_duplicates().dropna()

### Identify Column

In [39]:
# Try to detect target column automatically
print("Columns:", df.columns.tolist())

# Detect target column
possible_targets = ['label', 'attack', 'class', 'outcome', 'target']

target_column = None
for col in df.columns:
    if col.lower() in possible_targets:
        target_column = col
        break

if target_column is None:
    target_column = df.columns[-1]

print("Using target column:", target_column)
print(df[target_column].value_counts())

Columns: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE F

In [40]:
# confirm target values

print(df[target_column].value_counts())

Label
BENIGN      123280
PortScan     90819
Name: count, dtype: int64


### Encode

In [41]:
# Encode target FIRST
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(df[target_column])
y = to_categorical(y)

# Separate features
X = df.drop(columns=[target_column])

# Encode categorical features properly
X = pd.get_dummies(X)

# clean inf values
X = X.replace([np.inf, -np.inf], np.nan)
#X = X.fillna(0)
X = X.fillna(X.median())

# Split BEFORE scaling (avoids leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [42]:
# See all non natural numbers
print("Inf values:", np.isinf(X).sum().sum())
print("NaN values:", np.isnan(X).sum().sum())
print("Max value:", np.max(X.select_dtypes(include=[np.number]).values))

Inf values: 0
NaN values: 0
Max value: 2070000000.0


## Scale

In [43]:
# Scale numeric data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

num_features = X_train.shape[1]
num_classes = y.shape[1]

print("Features:", num_features)
print("Classes:", num_classes)

Features: 78
Classes: 2


### Split Train from test

## Model Creation

In [44]:
# Model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(num_features,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        10,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,010 (74.26 KB)

 Trainable params: 18,754 (73.26 KB)

 Non-trainable params: 256 (1.00 KB)

## Training

In [45]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
4282/4282 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.9960 - loss: 0.0154 - val_accuracy: 0.9991 - val_loss: 0.0037
Epoch 2/10
4282/4282 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.9986 - loss: 0.0060 - val_accuracy: 0.9992 - val_loss: 0.0029
Epoch 3/10
4282/4282 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.9989 - loss: 0.0044 - val_accuracy: 0.9990 - val_loss: 0.0041
Epoch 4/10
4282/4282 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.9991 - loss: 0.0041 - val_accuracy: 0.9995 - val_loss: 0.0066
Epoch 5/10
4282/4282 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9992 - loss: 0.0033 - val_accuracy: 0.9995 - val_loss: 0.0016
Epoch 6/10
4282/4282 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.9992 - loss: 0.0034 - val_accuracy: 0.9978 - val_loss: 0.0077
Epoch 7/10
4282/4282 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.9993 - loss: 0.0031 - val_accuracy: 0.9998 - val_loss: 0.0015
Epoch 8/10
4282/4282 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9994 - loss: 0.0023

## Evaluation

In [46]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")

1339/1339 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9998 - loss: 0.0014
Test Accuracy: 0.9998


## Setup

In [47]:
model.save("p4_model.h5")